# 06 — The three figures

**Reads:** `results/tables/` (written by notebooks 02 and 05).

**Writes:** `results/figures/fig1_class_experiment.html`, `fig2_growth.html`, `fig3_random_portfolios.html`.
When the website repository sits next to this one, the three files are also copied into the post's folder.

The figures are interactive Plotly pages embedded in the post as iframes, in the site's colours. Each carries a
title that says what is plotted, axis labels with units, a legend and a source line. Findings go in the post's
text, not in the figure.

In [1]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly_style import SITE, COLORS, base_layout, axis_style, source_note, save_html, save_html_controls

ROOT = Path.cwd().parent
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

S = json.load(open(TAB / "eval_settings.json"))
first_day, last_day = pd.Timestamp(S["first_day"]), pd.Timestamp(S["last_day"])
SPAN = f"{first_day:%b %Y} – {last_day:%b %Y}"
TOP = 150                                   # top margin: room for a legend that wraps on a phone
LINE = {"original": ("Class model", COLORS["post"]),
        "improved": ("Improved model", COLORS["pre"]),
        "universe": ("All 96 stocks, equal weight", COLORS["teal"]),
        "spy": ("SPY (S&P 500 ETF)", COLORS["grey"])}

## Figure 1 — the class experiment

One bar per team: the indicator portfolio's final value minus the intuition portfolio's, sorted from the team
where intuition won by most to the team where indicators won by most. The dashed line is the mean difference
and the band is its 95% bootstrap interval.

In [2]:
teams = pd.read_csv(TAB / "class_experiment_teams.csv").sort_values("difference").reset_index(drop=True)
cs = pd.read_csv(TAB / "class_experiment.csv", index_col="statistic")["value"]
mean_d, lo, hi = float(cs["mean_difference"]), float(cs["boot_ci95_lo"]), float(cs["boot_ci95_hi"])
teams["rank"] = np.arange(1, len(teams) + 1)
ahead = teams["difference"] > 0

fig = go.Figure()
for mask, name, color in [(~ahead, f"Intuition ahead ({int((~ahead).sum())} teams)", COLORS["post"]),
                          (ahead, f"Indicators ahead ({int(ahead.sum())} teams)", COLORS["pre"])]:
    t = teams[mask]
    fig.add_trace(go.Bar(x=t["rank"], y=t["difference"], name=name, marker_color=color,
                         customdata=t[["intuition", "indicators"]],
                         hovertemplate="Intuition: $%{customdata[0]:,.2f}<br>Indicators: $%{customdata[1]:,.2f}"
                                       "<br>Difference: $%{y:,.2f}<extra></extra>"))
x0, x1 = 0.4, len(teams) + 0.6
fig.add_trace(go.Scatter(x=[x0, x1, x1, x0], y=[lo, lo, hi, hi], fill="toself", mode="lines",
                         line=dict(width=0), fillcolor="rgba(209,154,58,0.28)", hoverinfo="skip",
                         name="95% bootstrap interval for the mean"))
fig.add_trace(go.Scatter(x=[x0, x1], y=[mean_d, mean_d], mode="lines", name="Mean difference",
                         line=dict(color=COLORS["ink"], width=2, dash="dash"),
                         hovertemplate="Mean difference: $%{y:,.2f}<extra></extra>"))
fig.update_layout(**base_layout("Indicators minus intuition, by team", height=540))
fig.update_layout(bargap=0.15, margin=dict(l=100, r=30, t=TOP, b=100),
                  xaxis=axis_style("Teams, sorted by the difference", showticklabels=False, ticks="", showgrid=False,
                                   range=[x0, x1]),
                  yaxis=axis_style("Indicators minus intuition ($)", tickprefix="$", tickformat=",.0f",
                                   zeroline=True, zerolinecolor=SITE["axis"], zerolinewidth=1))
source_note(fig, "AEDS 6400 class experiment, Fall 2026", y=-0.2)
save_html(fig, FIG / "fig1_class_experiment.html")
fig.show()

## Figure 2 — growth of $1 out of sample

Every line comes from the same backtest in notebook 05: ten stocks in equal amounts, rebalanced monthly,
before trading costs. The two benchmarks are SPY and all 96 stocks in equal amounts.

In [3]:
daily = pd.read_csv(TAB / "daily_returns.csv", index_col="date", parse_dates=True)
growth = (1 + daily).cumprod()
growth.loc[first_day] = 1.0                      # every line starts at $1 on the first entry date
growth = growth.sort_index()

fig = go.Figure()
for key in ["original", "improved", "universe", "spy"]:
    name, color = LINE[key]
    fig.add_trace(go.Scatter(x=growth.index, y=growth[key], mode="lines", name=name,
                             line=dict(color=color, width=2.4 if key in ("original", "improved") else 1.8,
                                       dash="dot" if key == "spy" else None),
                             hovertemplate=name + ": $%{y:.2f}<extra></extra>"))
fig.update_layout(**base_layout("Growth of $1 since January 2024", height=540))
fig.update_layout(hovermode="x unified", margin=dict(l=70, r=30, t=TOP, b=90),
                  xaxis=axis_style(None, hoverformat="%d %b %Y"),
                  yaxis=axis_style(f"Value of $1 invested on {first_day.day} {first_day:%b %Y}", tickprefix="$", tickformat=".2f"))
source_note(fig, "Yahoo Finance", y=-0.12)
save_html(fig, FIG / "fig2_growth.html")
fig.show()

## Figure 3 — the models against 10,000 random portfolios

The histogram is the result of portfolios that pick ten of the 96 stocks at random every month. The vertical
lines mark the two models and the two benchmarks. The control switches between two measures: the total return,
and the return per unit of risk (the average monthly return divided by its standard deviation, annualised).
`save_html_controls` writes the figure into a page with a plain `<select>`; each choice maps to a set of
visible traces and an axis.

In [4]:
rnd = pd.read_csv(TAB / "random_portfolios.csv", index_col="draw")
summary = pd.read_csv(TAB / "random_summary.csv", index_col="strategy")
MEASURES = {   # option -> (column of the draws, column of the strategy, percentile column, scale, axis title, suffix)
    "Total return": ("total_return", "total_return", "percentile_total_return", 100,
                     f"Total return, {SPAN} (%)", "%"),
    "Return per unit of risk": ("return_to_risk", "return_to_risk", "percentile_return_to_risk", 1,
                                "Annualised return divided by volatility, monthly returns", ""),
}

fig = go.Figure()
vis, lay = {}, {}
n_traces = 5 * len(MEASURES)
for m, (opt, (col, own, pct_col, scale, x_title, suffix)) in enumerate(MEASURES.items()):
    x = scale * rnd[col].to_numpy()
    marks = {k: scale * float(summary.loc[k, own]) for k in LINE}
    lo_edge, hi_edge = min(x.min(), min(marks.values())), max(x.max(), max(marks.values()))
    pad = 0.03 * (hi_edge - lo_edge)
    edges = np.linspace(lo_edge - pad, hi_edge + pad, 61)
    counts, _ = np.histogram(x, bins=edges)
    centres = (edges[:-1] + edges[1:]) / 2
    top = counts.max() * 1.05
    unit = "%" if suffix else ""
    fig.add_trace(go.Bar(x=centres, y=counts, width=edges[1] - edges[0], name="Random portfolios",
                         marker=dict(color=SITE["rule"], line=dict(color=SITE["axis"], width=0.5)),
                         hovertemplate="About %{x:.2f}" + unit + "<br>%{y:,} portfolios<extra></extra>"))
    for key in ["original", "improved", "universe", "spy"]:
        name, color = LINE[key]
        pct = float(summary.loc[key, pct_col])
        fig.add_trace(go.Scatter(x=[marks[key], marks[key]], y=[0, top], mode="lines", name=name,
                                 line=dict(color=color, width=2.6, dash="dot" if key == "spy" else None),
                                 hovertemplate=f"{name}: %{{x:.2f}}{unit}<br>ahead of {pct:.2f}% of random portfolios"
                                               "<extra></extra>"))
    vis[opt] = [m * 5 <= t < (m + 1) * 5 for t in range(n_traces)]
    lay[opt] = {"xaxis.title.text": x_title, "xaxis.ticksuffix": suffix,
                "xaxis.range": [float(edges[0]), float(edges[-1])], "yaxis.range": [0, float(top)]}

fig.update_layout(**base_layout("Models against 10,000 random portfolios", height=560))
fig.update_layout(bargap=0, margin=dict(l=70, r=30, t=TOP, b=110),
                  xaxis=axis_style(" "), yaxis=axis_style("Number of random portfolios", tickformat=","))
source_note(fig, "Yahoo Finance", y=-0.24)
save_html_controls(fig, FIG / "fig3_random_portfolios.html",
                   controls=[("measure", "Measure", list(MEASURES))], visibility=vis, layouts=lay)
fig.update_traces(visible=False)
for t in range(5):
    fig.data[t].visible = True
fig.update_layout(xaxis_title_text=lay["Total return"]["xaxis.title.text"], xaxis_ticksuffix="%")
fig.show()

## Copy into the post's folder

In [5]:
post = ROOT.parent / "Personal Website" / "blog" / "posts" / "post9"
if post.exists():
    for f in sorted(FIG.glob("fig*.html")):
        shutil.copy(f, post / f.name)
    print("copied to", post)
else:
    print("website folder not found next to this repository; figures are in", FIG)

copied to /sessions/rcw-01kwxbtwa664wpjxshwfkusr/mnt/UPenn/1st Semester/Computational Methods/Personal Website/blog/posts/post9
